# LlamaIndex Agents

**Domain:** Agentic AI  ·  **recommended addition**  ·  **runnable:** yes

A refresher on **agents in LlamaIndex** — the layer that turns an LLM plus a set of tools into something that *reasons, calls tools in a loop, and stops when it has an answer*. The signature LlamaIndex move: your **query engines (RAG pipelines) become tools**, so an agent can decide *when* to search your data, *which* index to hit, and how to combine that with other tools — instead of you hard-wiring a single retrieve-then-answer chain.

If you know the bare [[react]] loop, LlamaIndex agents are that loop made productized and wired into LlamaIndex's data/RAG stack. Compare with [[langchain]], [[crewai]], [[autogen]], and [[smolagents]] for the same idea in other framings.

## 1. What & Why

LlamaIndex started as a **data framework for RAG**: load documents, index them, query them. A plain query engine is a *fixed* pipeline — embed the question, retrieve top-k chunks, stuff them into a prompt, answer. That's great when every question is "look something up in my docs," and useless the moment a question needs *more than one* lookup, *arithmetic*, a *live API call*, or a *choice* between several data sources.

**Agents close that gap.** An agent wraps an LLM around a set of **tools** and runs a loop: the model reads the request, decides which tool to call (if any), you run it, the result goes back into context, and it repeats until the model produces a final answer. The decisive LlamaIndex feature is that **a query engine is just another tool** (`QueryEngineTool`). So an agent can hold three RAG indexes (10-Ks, support tickets, product docs) plus a calculator plus a web-search tool, and *route* each sub-question to the right one — dynamic RAG instead of static RAG.

**The problem it solves:**

- **Multi-step / multi-hop questions** — "Compare revenue growth in the 2022 and 2023 filings and tell me which grew faster" needs two retrievals and a comparison; a single query engine can't.
- **Routing over many sources** — with several indexes, the agent picks the relevant one(s) per question instead of you writing routing logic.
- **Tools beyond retrieval** — math, code execution, REST APIs, sending email — mixed freely with RAG tools.
- **Multi-agent orchestration** — `AgentWorkflow` runs several specialized agents that hand off to each other.

**When to reach for it:** your questions need more than one retrieval, a choice among data sources, or non-retrieval actions. **When *not* to:** if every query is a single lookup against one index, a plain `index.as_query_engine()` is cheaper, faster, and far more predictable than letting an LLM drive a loop. Don't pay for agency you don't use.

## 2. Mental Model

**An agent is a dispatcher with a phone book; query engines are entries in the phone book.**

A plain RAG query engine is a single hard-wired phone line — every question rings the same desk. A LlamaIndex agent is a switchboard operator: it reads the request, picks the right number from a directory of **tools** (some of which are entire RAG pipelines), places the call, listens to the answer, and decides whether to make another call or report back.

```
        user question
             │
             ▼
   ┌───────────────────────┐      tools (the phone book)
   │       AGENT (LLM)      │   ┌──────────────────────────┐
   │  ┌─────────────────┐  │   │ QueryEngineTool: 10-K RAG │
   │  │ think: which     │──┼──▶│ QueryEngineTool: tickets  │
   │  │ tool, what args? │  │   │ FunctionTool:  calculator │
   │  └─────────────────┘  │   │ FunctionTool:  web_search │
   │          ▲   │ call    │   └──────────────────────────┘
   │          │   ▼         │
   │     observation ◀──────┼──── tool result
   └──────────┴────────────┘
        loop until done → Final Answer
```

Two agent runtimes implement this loop:

- **`FunctionAgent`** — uses the model's **native function/tool-calling** API. The model emits a structured tool call; robust, the modern default for models that support it.
- **`ReActAgent`** — uses the **text-based [[react]]** prompt (`Thought / Action / Observation`). The fallback for models without native tool calling, and useful when you want the reasoning trace in plain text.

Same loop, two transports. Everything else — tools, memory, workflow orchestration — is shared.

## 3. Key Concepts

| Concept | What it is |
|---|---|
| **Tool** | A callable the agent may invoke, described by name + description + an auto-generated argument **schema**. The LLM reads the description to decide *whether* and *how* to call it. |
| **`FunctionTool`** | Wraps any Python function as a tool. `FunctionTool.from_defaults(fn=...)` infers the name, description (from the docstring), and JSON schema (from type hints) — so write a real docstring and type your args. |
| **`QueryEngineTool`** | Wraps a **query engine (a RAG pipeline) as a tool**. This is LlamaIndex's defining feature: the agent can *decide* when to query your data and which index to hit. |
| **`FunctionAgent`** | Agent runtime built on the model's **native tool-calling**. Preferred when the LLM supports it (most modern ones do). |
| **`ReActAgent`** | Agent runtime built on the **text-based ReAct** prompt. Use for models without native tool calling, or when you want the textual reasoning trace. |
| **`AgentWorkflow`** | Orchestrator for **one or more agents**. A single agent is the simple case; multiple agents with `can_handoff_to` enables multi-agent routing/hand-off. Runs are **async** (`await workflow.run(...)`). |
| **Context / memory** | A `Context` object carries conversation state and tool memory across turns, so a follow-up question remembers earlier ones. |
| **`Settings`** | Global defaults for `llm` and `embed_model`, so you don't pass them to every component. Set once at startup. |
| **Streaming events** | A run emits events (`AgentStream`, `ToolCall`, `ToolCallResult`) you can subscribe to for tokens and tool-trace observability. |

## 4. Setup

LlamaIndex is split into a small **core** plus per-integration packages (LLMs, embeddings, readers) you install only as needed:

```bash
pip install llama-index-core                 # tools, agents, workflows (no LLM)
pip install llama-index-llms-openai          # or -anthropic, -ollama, ...
pip install llama-index-embeddings-openai    # for QueryEngineTool / RAG
# Or the convenience metapackage that pulls common OpenAI defaults:
pip install llama-index
```

Then point it at a model:

```python
import os; os.environ["OPENAI_API_KEY"] = "sk-..."
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
Settings.llm = OpenAI(model="gpt-4o-mini")   # global default for every agent/query engine
```

**Examples 1 and 2 below need only `llama-index-core`** — no API key, no network — so they run in a fresh kernel offline. **Example 3** drives a real LLM and is gated behind an import + key check. The cell below reports what's available.

In [ ]:
# Environment check — Examples 1 & 2 need only llama-index-core (offline, no key).
import importlib.util as u, os

def _installed(name: str) -> bool:
    # find_spec raises if a *parent* package is missing, so guard it.
    try:
        return u.find_spec(name) is not None
    except ModuleNotFoundError:
        return False

has_core   = _installed("llama_index.core")
has_oai    = _installed("llama_index.llms.openai")
has_anthro = _installed("llama_index.llms.anthropic")
has_key    = bool(os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"))

if has_core:
    import llama_index.core as lic
    print("llama-index-core:", getattr(lic, "__version__", "?"))
else:
    print("llama-index-core: NOT installed  ->  pip install llama-index-core")

print("openai LLM integration   :", has_oai)
print("anthropic LLM integration:", has_anthro)
print("LLM API key present      :", has_key)
print()
print("Examples 1 & 2 run on core alone (offline, deterministic).")
print("Example 3 runs only with an LLM integration + key; otherwise it prints the call shape.")

## 5. Worked Examples

### Example 1 — A function becomes a tool

The atom of every LlamaIndex agent is the **tool**. `FunctionTool.from_defaults(fn=...)` turns a plain Python function into one: it reads the **name** from the function, the **description** from the docstring, and the **argument schema** from the type hints. That schema is exactly what the LLM sees when deciding whether and how to call the tool — which is why a clear docstring and real type hints matter. Here we build two tools, inspect the auto-generated schema, and call one directly (no LLM needed — a tool is just a callable).

In [ ]:
from llama_index.core.tools import FunctionTool

def multiply(a: int, b: int) -> int:
    """Multiply two integers and return the product."""
    return a * b

def lookup_capital(country: str) -> str:
    """Return the capital city of a given country name."""
    capitals = {"france": "Paris", "japan": "Tokyo", "kenya": "Nairobi"}
    return capitals.get(country.strip().lower(), f"unknown ({country})")

mul_tool = FunctionTool.from_defaults(fn=multiply)
cap_tool = FunctionTool.from_defaults(fn=lookup_capital)

# What the LLM actually sees about each tool:
for t in (mul_tool, cap_tool):
    print(f"name       : {t.metadata.name}")
    print(f"description: {t.metadata.description.splitlines()[0]}")
    print(f"arg schema : {t.metadata.get_parameters_dict()['properties']}")
    print("-" * 60)

# A tool is callable directly — returns a ToolOutput wrapping the raw value.
out = mul_tool(a=6, b=7)
print("multiply(6, 7) ->", out.raw_output, "  (ToolOutput.content:", repr(out.content) + ")")

### Example 2 — The agent loop, made explicit

`FunctionAgent`/`ReActAgent` need a real LLM, so to *run* the loop offline we stand in a deterministic "planner" for the model. This shows precisely what the agent runtime does on every turn: read state, **choose a tool and arguments**, dispatch through the `FunctionTool` interface, fold the observation back in, and repeat until done. The question — *"What is 3 times the number of letters in the capital of France?"* — needs two different tools chained, which is exactly what a plain query engine can't do and an agent can.

In [ ]:
# Reuse the FunctionTools from Example 1 as the agent's action space.
tools = {t.metadata.name: t for t in (mul_tool, cap_tool)}

def planner(state: dict) -> dict:
    """Stand-in for the LLM: inspects state, returns the next tool call or a final answer.
    A real FunctionAgent gets this exact decision from the model's tool-calling output."""
    if "capital" not in state:
        return {"tool": "lookup_capital", "args": {"country": "France"}, "save_as": "capital"}
    if "answer" not in state:
        n = len(state["capital"])                       # letters in 'Paris' -> 5
        return {"tool": "multiply", "args": {"a": 3, "b": n}, "save_as": "answer"}
    return {"final": f"3 x {len(state['capital'])} letters in {state['capital']} = {state['answer']}"}

def run_agent(planner, tools, max_steps=6):
    state = {}
    for step in range(1, max_steps + 1):
        decision = planner(state)
        if "final" in decision:
            return decision["final"]
        name, args = decision["tool"], decision["args"]
        result = tools[name](**args).raw_output         # dispatch through the FunctionTool
        state[decision["save_as"]] = result
        print(f"step {step}: call {name}({args}) -> {result}")
    return "Stopped: hit max_steps."

print("FINAL:", run_agent(planner, tools))

### Example 3 — A real `FunctionAgent` (gated)

With a real LLM the manual planner disappears: you hand the tools to a `FunctionAgent` and `await agent.run(...)`. The model itself decides the tool calls. This cell runs only if an LLM integration (e.g. `llama-index-llms-openai`) is installed **and** a key is set; otherwise it prints the canonical call shape so the notebook still executes top-to-bottom.

The same pattern wraps a RAG pipeline as a tool — the defining LlamaIndex capability:

```python
from llama_index.core.tools import QueryEngineTool
qe = index.as_query_engine()                       # any LlamaIndex query engine
rag_tool = QueryEngineTool.from_defaults(
    query_engine=qe,
    name="company_filings",
    description="Search the company's 10-K filings. Use for revenue/financial questions.",
)
agent = FunctionAgent(tools=[rag_tool, multiply_tool], llm=Settings.llm)
```

Now the agent *decides* when to search your data versus do arithmetic — dynamic RAG.

In [ ]:
import asyncio

if has_core and (has_oai or has_anthro) and has_key:
    from llama_index.core.agent.workflow import FunctionAgent
    if has_oai:
        from llama_index.llms.openai import OpenAI
        llm = OpenAI(model="gpt-4o-mini")
    else:
        from llama_index.llms.anthropic import Anthropic
        llm = Anthropic(model="claude-haiku-4-5")

    agent = FunctionAgent(
        tools=[mul_tool, cap_tool],
        llm=llm,
        system_prompt="You are a helpful assistant. Use the tools to compute answers.",
    )
    response = asyncio.run(
        agent.run("What is 3 times the number of letters in the capital of France?")
    )
    print(response)
else:
    print("Skipping live agent run (need an LLM integration package + API key).")
    print("With one configured, the call shape is:")
    print()
    print("    from llama_index.core.agent.workflow import FunctionAgent")
    print("    agent = FunctionAgent(tools=[mul_tool, cap_tool], llm=llm)")
    print("    response = await agent.run('What is 3 x the letters in France\\'s capital?')")
    print()
    print("The model picks lookup_capital -> multiply on its own; no manual planner.")

## 6. Gotchas & Pitfalls

- **Using an agent where a query engine would do.** If every question is one lookup against one index, `index.as_query_engine()` is cheaper, faster, and deterministic. An agent adds an LLM-driven control loop — latency, cost, and a chance of wrong tool choices — that you only want when questions genuinely branch or chain.
- **Weak tool descriptions and missing type hints.** The LLM routes purely on each tool's name, description, and argument schema. A `FunctionTool` with no docstring and untyped args gives the model nothing to reason about — it'll call the wrong tool or pass wrong arguments. Treat the docstring as the tool's prompt.
- **Forgetting runs are async.** `AgentWorkflow` / `FunctionAgent.run()` are coroutines. In a script use `asyncio.run(agent.run(...))`; in a notebook `await agent.run(...)`. Calling `.run()` and not awaiting it silently does nothing.
- **Version/API churn.** LlamaIndex's agent API has moved a lot: the old `OpenAIAgent` / `ReActAgent.from_tools(...)` (the "agent runner/worker") is superseded by the `llama_index.core.agent.workflow` classes (`FunctionAgent`, `ReActAgent`, `AgentWorkflow`). Pin your version and check which generation a tutorial targets before copying code.
- **The split-package install trap.** `llama-index-core` has **no LLM**. Importing an agent and running it raises `ImportError: llama-index-llms-openai not found` until you install an integration. Install core *plus* the LLM/embedding packages you actually use.
- **No step/cost ceiling.** As with any [[react]]-style loop, a confused agent can call tools forever. Set a max-iterations / timeout and watch token spend; query engines as tools can each be expensive.
- **Over-broad tool sets.** Ten vaguely-described tools (or three near-identical RAG indexes) make routing unreliable. Fewer tools with sharp, non-overlapping descriptions beat a big fuzzy toolbox.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-offs vs LlamaIndex agents |
|---|---|---|
| **LlamaIndex agent (this)** | RAG-centric apps where the agent must route across **multiple indexes / data sources** or mix retrieval with other tools | Best-in-class data/RAG ergonomics (`QueryEngineTool`); agent API has churned, so pin versions |
| **Plain LlamaIndex query engine** | Every question is a single lookup against one index | No LLM control loop — cheaper, faster, deterministic. Use this until you actually need branching/chaining |
| **Raw [[react]] loop** | Learning the mechanism; a tiny single-file agent | You hand-build parsing, stopping, memory. LlamaIndex gives you all that plus RAG tools out of the box |
| **[[langchain]] / [[langgraph]]** | General-purpose tool agents; LangGraph for explicit state-machine control & persistence | Broader integrations and graph control; weaker, more manual RAG-as-tool ergonomics than LlamaIndex |
| **[[crewai]] / [[autogen]]** | Multi-agent **role/conversation** workflows as the primary abstraction | Richer multi-agent framing; LlamaIndex's `AgentWorkflow` does hand-offs too but data-tooling is its strength |
| **[[smolagents]]** | Minimal code-writing agents (actions as Python code) | Tiny and code-action-first; far less RAG infrastructure |

**Rule of thumb:** reach for LlamaIndex agents when your agent's main job is to **reason over your own data across several sources**. If retrieval is a single fixed step, drop to a query engine; if the data layer is incidental and you need heavy orchestration/state, LangGraph or a multi-agent framework may fit better.

## 8. Resources

- **LlamaIndex — Agents module guide (official)** — https://docs.llamaindex.ai/en/stable/module_guides/deploying/agents/
- **LlamaIndex — Building an agent (tutorial, FunctionAgent + tools)** — https://docs.llamaindex.ai/en/stable/understanding/agent/
- **LlamaIndex — Agent / multi-agent workflows (`AgentWorkflow`)** — https://docs.llamaindex.ai/en/stable/understanding/agent/multi_agent/
- **`QueryEngineTool` — wrapping a RAG pipeline as a tool** — https://docs.llamaindex.ai/en/stable/module_guides/deploying/agents/tools/
- **LlamaIndex GitHub (source, examples, changelog for the agent API)** — https://github.com/run-llama/llama_index
- **ReAct prompting (the loop underneath `ReActAgent`)** — https://arxiv.org/abs/2210.03629

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def tool_from(fn, name=None, description=None):
    ...


def run_agent(planner, tools, max_steps=6):
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE